In [266]:
import os, sys, time
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
pd.options.mode.chained_assignment = None
import h5py
import sqlalchemy
from shapely import wkt
import geopandas as gpd
import seaborn as sns
from itertools import cycle, islice
import pyodbc
import warnings
import matplotlib.pyplot as plt
import psrcelmerpy
import numpy as np
from scipy.spatial import cKDTree

In [267]:
df_lu = pd.read_csv(r'C:\Workspace\displacement_index\parcels_urbansim.txt',
                   delim_whitespace=True)

# Load as a geodataframe
gdf_lu = gpd.GeoDataFrame(
    df_lu, geometry=gpd.points_from_xy(df_lu.xcoord_p, df_lu.ycoord_p))

crs = {'init' : 'EPSG:2285'}
gdf_lu.crs = crs
base_year = "2023"

parcel_geog = pd.read_sql_table('parcel_'+base_year+'_geography', 'sqlite:///N:/rtp_2026_2050/final_runs/sc_base_year_2023_final/soundcast/inputs/db/soundcast_inputs_2023.db')

In [ ]:
gdf_lu["geometry"]

0          POINT (1292255.144 162728.617)
1          POINT (1291832.241 164041.743)
2           POINT (1291594.615 164048.67)
3          POINT (1291539.635 164050.179)
4          POINT (1291479.355 164042.397)
                        ...              
1329923      POINT (1444396.3 305434.297)
1329924    POINT (1310124.174 305444.997)
1329925    POINT (1294709.295 305556.196)
1329926    POINT (1306089.393 288830.865)
1329927    POINT (1313226.373 286303.836)
Name: geometry, Length: 1329928, dtype: geometry

In [304]:
parcel_geog = parcel_geog.drop("geometry", axis=1)

KeyError: "['geometry'] not found in axis"

In [305]:
new_df_lu = gdf_lu.merge(parcel_geog,left_on='parcelid', right_on='ParcelID', how='left')

In [337]:
new_df_lu["GEOID20"]

0          5.303303e+14
1          5.303303e+14
2          5.303303e+14
3          5.303303e+14
4          5.303303e+14
               ...     
1329923    5.306105e+14
1329924    5.306105e+14
1329925    5.306105e+14
1329926    5.306105e+14
1329927    5.306105e+14
Name: GEOID20, Length: 1329928, dtype: float64

In [306]:
eg_conn = psrcelmerpy.ElmerGeoConn()
schools_gdf = eg_conn.read_geolayer('public_schools')
schools_gdf.columns
# schools_gdf[["object_id", "school", "geocoded_y", "geocoded_x", "nces_y", "nces_x", "Shape", "geometry"]]
schools_gdf.to_csv(r'C:\Workspace\displacement_index\displacement_index_current\11-Proximity-to-Civic-Infrastructure\public_schools.csv', index=False)

In [307]:
schools_gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [308]:
crs = {'init' : 'EPSG:2285'}
schools_gdf = schools_gdf.to_crs(crs)

In [309]:
schools_gdf.crs

<Projected CRS: EPSG:2285>
Name: NAD83 / Washington North (ftUS)
Axis Info [cartesian]:
- E[east]: Easting (US survey foot)
- N[north]: Northing (US survey foot)
Area of Use:
- name: United States (USA) - Washington - counties of Chelan; Clallam; Douglas; Ferry; Grant north of approximately 47°30'N; Island; Jefferson; King; Kitsap; Lincoln; Okanogan; Pend Oreille; San Juan; Skagit; Snohomish; Spokane; Stevens; Whatcom.
- bounds: (-124.79, 47.08, -117.02, 49.05)
Coordinate Operation:
- name: SPCS83 Washington North zone (US survey foot)
- method: Lambert Conic Conformal (2SP)
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [ ]:
school_district_gdf = eg_conn.read_geolayer('school_districts')
school_district_gdf = school_district_gdf[["lea_code", "short_name", "lea_name", "Shape", "geometry"]]


In [311]:
crs = {'init' : 'EPSG:2285'}
school_district_gdf = school_district_gdf.to_crs(crs)

In [312]:
school_district_gdf.crs

<Projected CRS: EPSG:2285>
Name: NAD83 / Washington North (ftUS)
Axis Info [cartesian]:
- E[east]: Easting (US survey foot)
- N[north]: Northing (US survey foot)
Area of Use:
- name: United States (USA) - Washington - counties of Chelan; Clallam; Douglas; Ferry; Grant north of approximately 47°30'N; Island; Jefferson; King; Kitsap; Lincoln; Okanogan; Pend Oreille; San Juan; Skagit; Snohomish; Spokane; Stevens; Whatcom.
- bounds: (-124.79, 47.08, -117.02, 49.05)
Coordinate Operation:
- name: SPCS83 Washington North zone (US survey foot)
- method: Lambert Conic Conformal (2SP)
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [313]:
school_district_gdf.total_bounds

array([1095631.85983406,  -97406.58393161, 1622341.8920418 ,
        506798.53114656])

In [314]:
school_district_gdf = school_district_gdf.copy()
print(school_district_gdf.total_bounds)

[1095631.85983406  -97406.58393161 1622341.8920418   506798.53114656]


In [ ]:
school_district_gdf = school_district_gdf.copy()
new_df_lu =  new_df_lu.copy()
print(new_df_lu.total_bounds)
print(school_district_gdf.total_bounds)

print(len(new_df_lu))
print(len(school_district_gdf))


parcel_school_district = gpd.sjoin(
    new_df_lu[['Census2020BlockGroup','parcelid', 'geometry']],  # keep only what you need
    school_district_gdf[['lea_name', 'geometry']],  # keep only what you need
    how='left',     
    predicate='intersects'   # or 'intersects'
)
parcel_school_district

[1099777.18725    -93023.8403113 1610615.20787    476785.443993 ]
[1095631.85983406  -97406.58393161 1622341.8920418   506798.53114656]
1329928
53


,Census2020BlockGroup,parcelid,geometry,hh_p,index_right,lea_name
0,5.303303e+11,1,POINT (1292255.144 162728.617),0,6.0,Renton School District
1,5.303303e+11,2,POINT (1291832.241 164041.743),0,6.0,Renton School District
2,5.303303e+11,3,POINT (1291594.615 164048.67),0,6.0,Renton School District
3,5.303303e+11,4,POINT (1291539.635 164050.179),0,6.0,Renton School District
4,5.303303e+11,5,POINT (1291479.355 164042.397),0,6.0,Renton School District
...,...,...,...,...,...,...
1329923,5.306105e+11,1329924,POINT (1444396.3 305434.297),0,49.0,Sultan School District
1329924,5.306105e+11,1329925,POINT (1310124.174 305444.997),0,18.0,Northshore School District
1329925,5.306105e+11,1329926,POINT (1294709.295 305556.196),0,42.0,Edmonds School District
1329926,5.306105e+11,1329927,POINT (1306089.393 288830.865),0,18.0,Northshore School District


In [316]:
missing = parcel_school_district['lea_name'].isna().sum()
print(missing)

3229


In [317]:
parcel_dict = {
    name: group
    for name, group in parcel_school_district.groupby("lea_name")
}

In [318]:
parcel_sum = 0
for district in parcel_dict.keys():
    parcel_sum += len(parcel_dict[district])
print(parcel_sum)

1326699


In [110]:
parcel_school_district.to_csv(r'C:\Workspace\displacement_index\displacement_index_current\11-Proximity-to-Civic-Infrastructure\parcel_school_district.csv', index=False)

In [109]:
school_district_gdf.to_csv(r'C:\Workspace\displacement_index\displacement_index_current\11-Proximity-to-Civic-Infrastructure\school_districts.csv', index=False)

In [319]:
school_district_gdf['lea_code'] = school_district_gdf['lea_code'].astype(str)
schools_gdf['lea_code'] = schools_gdf['lea_code'].astype(str)
school_merge = school_district_gdf.merge(schools_gdf, left_on='lea_code', right_on='lea_code', how='left')

In [320]:
school_merge.columns

Index(['lea_code', 'short_name', 'lea_name_x', 'Shape_x', 'geometry_x',
       'OBJECTID', 'object_id', 'school_code', 'school', 'single_address',
       'esd_code', 'esd_name', 'lea_name_y', 'school_category', 'ayp_code',
       'grade_category', 'principal', 'phone', 'email', 'lowest_grade',
       'highest_grade', 'mailing_address', 'congression', 'legislative',
       'county', 'geocoded_y', 'geocoded_x', 'nces_y', 'nces_x', 'adjusted_l',
       'SDE_STATE_ID', 'Shape_y', 'geometry_y'],
      dtype='object')

In [322]:
school_merge = school_merge[["lea_code", "short_name", "lea_name_x", "county", "geometry_x", "school_code", "school", "school_category", "grade_category", "geometry_y"]]

In [323]:
school_merge = school_merge.rename(columns={
    "geometry_x": "district_geometry", 
    "geometry_y": "school_geometry"})

In [324]:
school_merge

,lea_code,short_name,lea_name_x,county,district_geometry,school_code,school,school_category,grade_category,school_geometry
0,17001,Seattle,Seattle Public Schools,King,"POLYGON ((1247383.762 271911.64, 1247296.394 2...",2450,Daniel Bagley Elementary School,"Public School, Regular School",Elementary School,POINT (1268752.026 254011.927)
1,17001,Seattle,Seattle Public Schools,King,"POLYGON ((1247383.762 271911.64, 1247296.394 2...",2199,Concord International School,"Public School, Regular School",Elementary School,POINT (1272009.138 194561.023)
2,17001,Seattle,Seattle Public Schools,King,"POLYGON ((1247383.762 271911.64, 1247296.394 2...",3974,Thornton Creek Elementary School,"Public School, Alternative School",Elementary School,POINT (1282965.33 253340.115)
3,17001,Seattle,Seattle Public Schools,King,"POLYGON ((1247383.762 271911.64, 1247296.394 2...",2285,Roosevelt High School,"Public School, Regular School",High School,POINT (1275977.162 250559.284)
4,17001,Seattle,Seattle Public Schools,King,"POLYGON ((1247383.762 271911.64, 1247296.394 2...",3218,North Beach Elementary School,"Public School, Regular School",Elementary School,POINT (1257805.837 257302.086)
...,...,...,...,...,...,...,...,...,...,...
1078,31401,Stanwood-Camano,Stanwood-Camano School District,Snohomish,"POLYGON ((1246446.841 477516.376, 1244843.126 ...",4364,Twin City Elementary School,"Public School, Regular School",Elementary School,POINT (1275985.789 453850.219)
1079,31401,Stanwood-Camano,Stanwood-Camano School District,Snohomish,"POLYGON ((1246446.841 477516.376, 1244843.126 ...",3125,Stanwood Elementary School,"Public School, Regular School",Elementary School,POINT (1265293.341 457928.645)
1080,31401,Stanwood-Camano,Stanwood-Camano School District,Snohomish,"POLYGON ((1246446.841 477516.376, 1244843.126 ...",4512,Port Susan Middle School,"Public School, Regular School",Middle School,POINT (1274323.749 454867.719)
1081,31401,Stanwood-Camano,Stanwood-Camano School District,Snohomish,"POLYGON ((1246446.841 477516.376, 1244843.126 ...",4513,Cedarhome Elementary School,"Public School, Regular School",Elementary School,POINT (1277358.083 459260.02)


In [325]:
school_dict = {
    district: group
    for district, group in school_merge.groupby("lea_name_x")
}


In [326]:
school_dict["Arlington School District"]

,lea_code,short_name,lea_name_x,county,district_geometry,school_code,school,school_category,grade_category,school_geometry
987,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",4573,Pioneer Elementary School (Arlington),"Public School, Regular School",Elementary School,POINT (1326297.119 427802.451)
988,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",2277,Arlington Special Educ School,Public School,PK-12,POINT (1326133.353 438805.808)
989,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",3124,Post Middle School,"Public School, Regular School",Middle School,POINT (1328729.118 439035.415)
990,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",4154,Presidents Elementary School,"Public School, Regular School",Elementary School,POINT (1326552.714 438706.85)
991,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",4436,Kent Prairie Elementary School,"Public School, Regular School",Elementary School,POINT (1326675.254 434363.215)
992,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",2523,Arlington High School,"Public School, Regular School",High School,POINT (1327667.168 427998.484)
993,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",5495,Arlington Open Doors,"Public School, Re-Engagement School",High School,POINT (1314122.418 423451.741)
994,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",4287,Weston High School,"Public School, Alternative School",High School,POINT (1314122.418 423451.741)
995,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",5010,Haller Middle School,"Public School, Regular School",Middle School,POINT (1326358.209 437287.23)
996,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",1714,Stillaguamish Valley Learning Center,"Public School, Alternative School",Other,POINT (1328017.726 438251.895)


In [328]:
def find_nearest(gdA, gdB):
    """ Find nearest value between two geodataframes.
        Returns "dist" for distance between nearest points.
    """

    nA = np.array(list(gdA.geometry.apply(lambda x: (x.x, x.y))))
    nB = np.array(list(gdB.geometry.apply(lambda x: (x.x, x.y))))
    btree = cKDTree(nB)
    dist, idx = btree.query(nA, k=1)
    gdB_nearest = gdB.iloc[idx].drop(columns="geometry").reset_index(drop=True)
    gdf = pd.concat(
        [
            gdA.reset_index(drop=True),
            gdB_nearest,
            pd.Series(dist, name='dist')
        ], 
        axis=1)

    return gdf

In [302]:
parcel_dict = parcel_dict.copy()
school_dict = school_dict.copy()
parcel_dict["Arlington School District"] = parcel_dict["Arlington School District"].rename(columns={'geometry_x': 'geometry'})
school_dict["Arlington School District"] = school_dict["Arlington School District"].rename(columns={'school_geometry': 'geometry'})
parcel_dict["Arlington School District"] = parcel_dict["Arlington School District"].set_geometry("geometry")
school_dict["Arlington School District"] = school_dict["Arlington School District"].set_geometry("geometry")

school_dict["Arlington School District"] = school_dict["Arlington School District"].set_crs(epsg=4326, allow_override=True)
parcel_dict["Arlington School District"] = parcel_dict["Arlington School District"].set_crs(epsg=4326, allow_override=True)

school_dict["Arlington School District"] = school_dict["Arlington School District"].to_crs(epsg=2285)
parcel_dict["Arlington School District"] = parcel_dict["Arlington School District"].to_crs(epsg=2285)

find_nearest(parcel_dict["Arlington School District"], school_dict["Arlington School District"])

ValueError: data must be finite, check for nan or inf values

In [329]:
results = []
for district_name in parcel_dict.keys():
    print(district_name)
    parcels = parcel_dict[district_name].copy()
    schools = school_dict[district_name].copy()
    
    parcels = parcels.rename(columns={'geometry_x': 'geometry'})
    schools = schools.rename(columns={'school_geometry': 'geometry'})

    # Make sure geometry column is active
    parcels = parcels.set_geometry("geometry")
    schools = schools.set_geometry("geometry")
    
    # Ensure CRS matches
    schools = schools.to_crs(parcels.crs)
    
    # Compute nearest
    nearest_df = find_nearest(parcels, schools)
    
    # Collect
    results.append(nearest_df)

Arlington School District
Auburn School District
Bainbridge Island School District
Bellevue School District
Bethel School District
Bremerton School District
Carbonado School District
Central Kitsap School District
Clover Park School District
Darrington School District
Dieringer School District
Eatonville School District
Edmonds School District
Enumclaw School District
Everett School District
Federal Way School District
Fife School District
Franklin Pierce School District
Granite Falls School District
Highline School District
Index School District
Issaquah School District
Kent School District
Lake Stevens School District
Lake Washington School District
Lakewood School District
Marysville School District
Mercer Island School District
Monroe School District
Mukilteo School District
North Kitsap School District
Northshore School District
Orting School District
Peninsula School District
Puyallup School District
Renton School District
Riverview School District
Seattle Public Schools
Shorelin

In [330]:
all_nearest_df = pd.concat(results, ignore_index=True)
all_nearest_df

,Census2020BlockGroup,parcelid,geometry,hh_p,index_right,lea_name,lea_code,short_name,lea_name_x,county,district_geometry,school_code,school,school_category,grade_category,dist
0,5.306105e+11,1062029,POINT (1325224.243 438413.283),0,43.0,Arlington School District,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",2277,Arlington Special Educ School,Public School,PK-12,990.230327
1,5.306105e+11,1062030,POINT (1325284.222 438411.697),0,43.0,Arlington School District,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",2277,Arlington Special Educ School,Public School,PK-12,936.133421
2,5.306105e+11,1062031,POINT (1325264.447 438336.327),0,43.0,Arlington School District,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",2277,Arlington Special Educ School,Public School,PK-12,987.627691
3,5.306105e+11,1062032,POINT (1325262.742 438279.495),0,43.0,Arlington School District,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",2277,Arlington Special Educ School,Public School,PK-12,1017.334091
4,5.306105e+11,1062033,POINT (1325247.916 438169.773),0,43.0,Arlington School District,31016,Arlington,Arlington School District,Snohomish,"POLYGON ((1309152.907 476285.658, 1306811.535 ...",2277,Arlington Special Educ School,Public School,PK-12,1090.201318
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1326694,5.305307e+11,1053553,POINT (1316396.678 56390.587),1,37.0,White River School District,27416,White River,White River School District,Pierce,"POLYGON ((1317390.685 80729.974, 1317265.72 80...",4309,Foothills Elementary School,"Public School, Regular School",Elementary School,6836.515029
1326695,5.305307e+11,1053554,POINT (1316385.696 56480.434),1,37.0,White River School District,27416,White River,White River School District,Pierce,"POLYGON ((1317390.685 80729.974, 1317265.72 80...",4309,Foothills Elementary School,"Public School, Regular School",Elementary School,6783.297478
1326696,5.305307e+11,1053555,POINT (1316423.672 56580.798),1,37.0,White River School District,27416,White River,White River School District,Pierce,"POLYGON ((1317390.685 80729.974, 1317265.72 80...",4309,Foothills Elementary School,"Public School, Regular School",Elementary School,6687.521034
1326697,5.305307e+11,1053556,POINT (1316441.219 56729.259),1,37.0,White River School District,27416,White River,White River School District,Pierce,"POLYGON ((1317390.685 80729.974, 1317265.72 80...",4309,Foothills Elementary School,"Public School, Regular School",Elementary School,6575.427462


In [331]:
def weighted_avg(df, val_col, wt_col, agg_col):
    """ Returns weighted average for specified aggregation. 
        
        Parameters
    ----------
    df : Pandas DataFrame 
    val_col: column name of the value being averaged
    wt_col: weight column name
    agg_col: column to be used for aggregation
    ----------
    """
    df = df.copy()
    df['wt_tot'] = df[val_col] * df[wt_col]
    
    # Aggregate sums for each group
    df_agg = df.groupby(agg_col)[[wt_col, 'wt_tot']].sum()
    
    # Compute weighted average
    df_agg['wt_avg'] = df_agg['wt_tot'] / df_agg[wt_col]
    
    return df_agg

In [332]:
all_nearest_df['miles'] = all_nearest_df['dist'] / 5280.0
tract_output_df = weighted_avg(all_nearest_df, val_col='miles', wt_col='hh_p', agg_col='Census2020BlockGroup').reset_index()

In [334]:
tract_output_df.to_csv(r'C:\Workspace\displacement_index\displacement_index_current\11-Proximity-to-Civic-Infrastructure\pop_avg_dist_to_schools.csv')